# Bài 6 (Trang 45) - Groceries Dataset
## Câu 1: a) Tìm tập phổ biến bằng Apriori với từng min_sup và ghi nhận thời gian

**Sinh viên:** Trần Đình Võ - **MSSV:** 6451071088

In [1]:
HO_TEN = "Trần Đình Võ"
MSSV = "6451071088"
print(f"Họ tên: {HO_TEN} - MSSV: {MSSV}")

Họ tên: Trần Đình Võ - MSSV: 6451071088


### Import thư viện và đọc dữ liệu

In [2]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth
%matplotlib inline

with open("../Dataset/groceries.csv", encoding="utf-8") as f:
    transactions = [line.strip().split(",") for line in f if line.strip()]

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)
print("Số giao dịch:", df.shape[0], "- Số items:", df.shape[1])

# Thí nghiệm 1: min_sup = 0.01 -> 0.1 (bước 0.01)
supports = [round(0.01 * i, 2) for i in range(1, 11)]
# Thí nghiệm 2: min_sup rất nhỏ = 0.03% -> 0.1%
# (0.01%, 0.02% tương ứng support count 1-2 giao dịch -> số tập phổ biến bùng nổ tổ hợp, không chạy được)
supports_small = [0.0003, 0.0004, 0.0005, 0.0007, 0.001]


def benchmark(algo, sups):
    times, counts = [], []
    for s in sups:
        start = time.time()
        fi = algo(df, min_support=s, use_colnames=True)
        times.append(time.time() - start)
        counts.append(len(fi))
    return pd.DataFrame({"min_sup": sups, "so_tap_pho_bien": counts,
                         "thoi_gian_s": [round(t, 4) for t in times]})

Số giao dịch: 9835 - Số items: 169


### Câu 1: a) Tìm tập phổ biến bằng Apriori với từng min_sup và ghi nhận thời gian

In [3]:
apriori_result = benchmark(apriori, supports)
apriori_result_small = benchmark(apriori, supports_small)

In [4]:
print('Thí nghiệm 1: min_sup 0.01 -> 0.1')
apriori_result

Thí nghiệm 1: min_sup 0.01 -> 0.1


,min_sup,so_tap_pho_bien,thoi_gian_s
0,0.01,333,0.1264
1,0.02,122,0.0531
2,0.03,63,0.0318
3,0.04,41,0.0194
4,0.05,31,0.0156
5,0.06,21,0.0107
6,0.07,19,0.0091
7,0.08,13,0.0063
8,0.09,10,0.0057
9,0.10,8,0.0050


In [5]:
print('Thí nghiệm 2: min_sup 0.03% -> 0.1%')
apriori_result_small

Thí nghiệm 2: min_sup 0.03% -> 0.1%


,min_sup,so_tap_pho_bien,thoi_gian_s
0,0.0003,166682,72.3318
1,0.0004,80388,18.3050
2,0.0005,49580,10.5408
3,0.0007,25570,5.7778
4,0.0010,13492,3.2354


In [6]:
# Ví dụ: các tập phổ biến với min_sup = 0.05
apriori(df, min_support=0.05, use_colnames=True).sort_values('support', ascending=False)

,support,itemsets
26,0.255516,frozenset({whole milk})
15,0.193493,frozenset({other vegetables})
19,0.183935,frozenset({rolls/buns})
23,0.174377,frozenset({soda})
27,0.139502,frozenset({yogurt})
2,0.110524,frozenset({bottled water})
20,0.108998,frozenset({root vegetables})
24,0.104931,frozenset({tropical fruit})
22,0.098526,frozenset({shopping bags})
21,0.093950,frozenset({sausage})


### Nhận xét
- Khi min_sup giảm, số tập phổ biến tăng rất nhanh (min_sup 0.1 chỉ có 8 tập, 0.01 có 333 tập; 0.03% có hơn 160 nghìn tập) và thời gian chạy Apriori tăng mạnh.